# Modeling

Tahap pemodelan (*Modeling*) berfokus pada eksplorasi dan penentuan parameter optimal serta pelatihan model klasterisasi. Model yang dilatih disimpan ke direktori `artifact/3_modeling/` untuk kemudian dievaluasi dan dipilih pada tahap *Evaluation*.


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
from config import (
    PREPARED_REGENCIES_CSV,
    FEATURE_SELECTION_JSON,
    KMEANS_MODEL_PKL,
    AGGLOMERATIVE_MODEL_PKL,
    MODELING_DIR
)


In [ ]:
K_MIN = 2
K_MAX = 10
FALLBACK_K = 4
RANDOM_STATE = 42


In [ ]:
print(f"Parameter K_MIN        : {K_MIN}")
print(f"Parameter K_MAX        : {K_MAX}")
print(f"Parameter FALLBACK_K   : {FALLBACK_K}")
print(f"Parameter RANDOM_STATE : {RANDOM_STATE}")


In [ ]:
df_reg = pd.read_csv(PREPARED_REGENCIES_CSV)

feat_config = json.load(open(FEATURE_SELECTION_JSON, 'r', encoding='utf-8'))
scaled_cols = feat_config.get('scaled_feature_columns', [c for c in df_reg.columns if c.startswith('scaled_')])

X = df_reg[scaled_cols].values
print(f'Dimensi data latih: {X.shape}')


## Penentuan Parameter K Optimal

In [ ]:
k_range = list(range(K_MIN, K_MAX + 1))


### K-Means (Elbow Method / WCSS)

In [ ]:
wcss_km = []
for k in k_range:
    km_test = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
    km_test.fit(X)
    wcss_km.append(km_test.inertia_)

kn_km = KneeLocator(k_range, wcss_km, curve='convex', direction='decreasing')

if kn_km.knee is not None and kn_km.knee in k_range:
    optimal_k_km = int(kn_km.knee)
else:
    optimal_k_km = FALLBACK_K

print(f"Optimal K (K-Means) : {optimal_k_km}")


### Agglomerative (Silhouette Score Maximization)

In [ ]:
sil_scores_agg = []

for k in k_range:
    agg_test = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg_test.fit(X)
    score = silhouette_score(X, agg_test.labels_)
    sil_scores_agg.append(score)

best_agg_idx = int(np.argmax(sil_scores_agg))
optimal_k_agg = k_range[best_agg_idx]

linkage_matrix = linkage(X, method='ward')
print(f"Optimal K (Agglomerative) : {optimal_k_agg}")


### Visualisasi Penentuan Parameter

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Plot Elbow K-Means
axes[0].plot(k_range, wcss_km, marker='o')
axes[0].axvline(x=optimal_k_km, color='red', linestyle='--', label=f'Optimal K = {optimal_k_km}')
axes[0].set_title('K-Means: Elbow Method (WCSS)')
axes[0].set_xlabel('Jumlah Klaster (K)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

# Plot Silhouette Agglomerative
axes[1].plot(k_range, sil_scores_agg, marker='o')
axes[1].axvline(x=optimal_k_agg, color='red', linestyle='--', label=f'Optimal K = {optimal_k_agg}')
axes[1].set_title('Agglomerative: Silhouette Analysis')
axes[1].set_xlabel('Jumlah Klaster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

# Plot Dendrogram
dendrogram(linkage_matrix, ax=axes[2], truncate_mode='level', p=5, no_labels=True)
axes[2].set_title('Agglomerative: Dendrogram')
axes[2].set_xlabel('Klaster Gabungan')
axes[2].set_ylabel('Jarak Ward')

plt.tight_layout()
plt.show()


## Pelatihan & Penyimpanan Model Individual

In [ ]:
os.makedirs(os.path.dirname(KMEANS_MODEL_PKL), exist_ok=True)

kmeans_model = KMeans(n_clusters=optimal_k_km, random_state=RANDOM_STATE, n_init=10)
kmeans_model.fit(X)
pickle.dump(kmeans_model, open(KMEANS_MODEL_PKL, 'wb'))
print(f'Model K-Means (K={optimal_k_km}) tersimpan di: {KMEANS_MODEL_PKL}')

agg_model = AgglomerativeClustering(n_clusters=optimal_k_agg)
agg_model.fit(X)
pickle.dump(agg_model, open(AGGLOMERATIVE_MODEL_PKL, 'wb'))
print(f'Model Agglomerative (K={optimal_k_agg}) tersimpan di: {AGGLOMERATIVE_MODEL_PKL}')
